# Operations

**Prerequisites:** `01_what_memory_is.ipynb` and `02_stores.ipynb` from this chapter.

Notebook 2 left four stores behind — a JSON file, two SQLite databases, a Chroma collection, and five edges in a Neo4j instance — each one chosen by what would be asked of it. This notebook is about the verbs.

There are five: **write, update, read, retrieve, forget.** Every store appears to have them, which makes them look like an interface. They are not. Only the *signatures* are portable, and underneath:

- `write` on a graph needs a model call that no other store wants, because a graph does not accept sentences
- `update` on a vector index is a delete and an insert, because there is nothing inside a vector to edit
- `retrieve` does not exist on a dict at all
- `forget` on a graph is usually the wrong operation, and the right one is an end date

Most of this notebook is `retrieve`, because that is where all the difficulty is: ranking more candidates than fit in a prompt, deciding which are worth reading in full, and telling *this memory has not been needed in months* apart from *this memory is no longer true*.

**By the end you'll have:**
- Separated **read** from **retrieve**, and found that only one of the four stores ranks at all
- Timed the same verb across four stores, and hit one that cannot be called from a sentence without a model in the way
- Watched one sentence produce **two** graph edges, and a perfectly good memory produce none
- Proved a vector update is a replacement rather than an edit, by watching the same entry move for one phrase and not another
- Measured where a floor belongs instead of picking one, using questions you know the store cannot answer
- Separated three things that all look like time: reinforcement, validity, and the write date
- Watched the same query return a different third place depending on what the agent had been reading
- Put frequency **inside** lambda rather than beside it, so it stretches the clock instead of competing with recency
- Measured both weights against questions with known answers, and watched one memory take over every query when the reinforcement weight is too high
- Built two-stage recall only after showing it should not exist for one-line memories — and fixed four things in it that looked right and were not
- Given the write path a decision, so `update` has something that calls it and the same fact does not end up in the journal five times
- Separated decay from deletion, made eviction opt-in, and ended on the store where forgetting is neither

## Setup

Everything here opens what notebook 2 left on disk and in Neo4j. Nothing is rebuilt, and the notebook puts back anything it changes, so it can be run twice.

In [1]:
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(Path.cwd().parent.parent / chapter))

from stores import KeyValue, Records, Journal, Graph

STORE = Path("store")

profile = KeyValue(STORE / "profile.json")
records = Records(STORE / "records.db")
journal = Journal(STORE / "journal")
org = Graph()

print("profile:", profile.data)
print("records:", len(records), "rows")
print("journal:", len(journal), "entries")
print("graph  :", len(org), "edges")

profile: {'timezone': 'Asia/Kolkata', 'examples': 'Python'}
records: 6 rows
journal: 6 entries
graph  : 6 edges


## `read` and `retrieve` are not the same verb

Both hand back a memory, which is why they get treated as one operation with two names. The difference decides how much of this notebook applies to which store.

A **read** is a lookup. You know what you want, you name it, and you get exactly that or nothing.

A **retrieve** is a ranked guess. You cannot name what you want, so you describe it, and what comes back is an ordering over things that might do.

Notebook 2 ran both without separating them:

In [2]:
print("read     ->", repr(profile.read("timezone")))
print("read     ->", repr(profile.read("shell")), "  <- a read can say no")
print()

question = "checkout is timing out and the database seems to be blocking"
for score, entry_id, text in journal.find_like(question, k=3):
    print(f"retrieve -> {score:.4f}  #{entry_id}  {text[:54]}")

read     -> 'Asia/Kolkata'
read     -> None   <- a read can say no



retrieve -> 0.6670  #3  The migration held a lock on the users table for about
retrieve -> 0.6496  #1  The pool ran dry under peak traffic; raising max conne
retrieve -> 0.6263  #4  Billing runs its own migrations on a separate schedule


Two differences, and both matter later.

**A read is exact; a retrieve is ordered.** `timezone` has one right answer and the dict has it. The question about timing out has no right answer stored anywhere — it has entries that resemble it more or less, and the numbers are the *ranking*, not a verdict.

**A read can say no; a retrieve cannot.** `read("shell")` returned `None`. Ask the vector index for three and it returns three, always, whatever is in there. That property is the source of most of the work in this notebook.

Which raises the question of which stores actually retrieve. It is easy to assume all four do something ranked, so it is worth watching the other two answer:

In [3]:
for row in records.recent(status="failed", limit=2):
    print("Records ->", row["at"], row["service"], row["text"])

print()
print("Graph   ->", Graph.phrase(org.path("Meera", "checkout-api")))

Records -> 2026-08-19 billing-api Rolled back. Error rate hit 4 percent.
Records -> 2026-08-18 billing-api Schema cache error.

Graph   -> Meera owns billing-api -> checkout-api calls billing-api


No scores anywhere, and none missing. `Records` filtered and ordered by a column; ask it twice and the same rows come back in the same order, because `ORDER BY at DESC` is not an opinion. `Graph` walked from one node to another and returned the chain it found; there was no second-best path being quietly discarded.

**Only the journal ranks.** The other three answer their questions exactly — a dict by name, SQL by predicate, a graph by traversal — and an exact answer has nothing to rank. That is not a limitation of those three; it is what notebook 2 chose them for.

So from here the notebook splits. `write`, `update` and `forget` are worth walking across all four stores, because each one distorts them differently. `retrieve` is a journal problem, and it takes the rest of the notebook.

## `write`: one verb, four different amounts of work

Same call shape everywhere. Watch what each one costs:

In [4]:
note = ("2026-08-26", "billing-api", "concluded",
        "The schema cache only refreshes on NOTIFY, so a migration that skips it leaves "
        "PostgREST serving a stale column list.")

t = time.perf_counter()
profile.write("editor", "vim")
print("KeyValue.write  %7.1f ms   dict, then rewrite one JSON file" % ((time.perf_counter() - t) * 1000))

t = time.perf_counter()
record_id = records.write("2026-08-26", "incident", "resolved", "billing-api", "Stale schema cache.")
print("Records.write   %7.1f ms   one INSERT against a fixed schema" % ((time.perf_counter() - t) * 1000))

t = time.perf_counter()
[entry_id] = journal.write([note])
print("Journal.write   %7.1f ms   INSERT, FTS5 index, and an embedding call over the network"
      % ((time.perf_counter() - t) * 1000))

KeyValue.write      1.1 ms   dict, then rewrite one JSON file
Records.write       4.0 ms   one INSERT against a fixed schema


Journal.write     733.8 ms   INSERT, FTS5 index, and an embedding call over the network


Two local writes in single-digit milliseconds, and one that leaves the machine and takes the better part of a second. The journal write is three writes wearing one name: a row in SQLite, an entry in the FTS5 index, and a vector that only exists once a model has produced it. It is the first place in this chapter where saving a memory can *fail because a service was down*.

The fourth store cannot be written to at all from what we have. `Graph.write` takes a triple:

```python
org.write("Meera", "owns", "billing-api")
```

and what actually arrives is a sentence. *"Devi took over billing-api from Meera on the 20th"* is not a triple, and no amount of string handling turns it into one. Something has to read it and decide what the entities are, which relation holds, and which way it points — and that something is a model.

`03_tools` already built the machinery: put the permitted relations in the schema with `Literal`, and the model picks from them rather than inventing a vocabulary.

In [5]:
from typing import Literal, Optional

from pydantic import BaseModel, Field

from llm import extract


class Edge(BaseModel):
    subject: str = Field(description="the thing the statement is about")
    relation: Literal["owns", "calls", "decided", "reports_to"]
    object: str = Field(description="the thing at the other end")


class Change(BaseModel):
    """What a sentence does to a graph: some edges begin, some stop being true."""

    at: Optional[str] = Field(default=None, description="YYYY-MM-DD of the change, if stated")
    starts: list[Edge] = Field(default_factory=list, description="relationships that become true")
    ends: list[Edge] = Field(default_factory=list, description="relationships that stop being true")


PROMPT = """Turn this statement into changes to a knowledge graph.

Statement: {s}

Use only the relations in the schema. `ends` is for relationships the statement says have
stopped: someone handing something over ends their own edge as well as starting the new one.
Leave a list empty rather than inventing an edge."""

for sentence in ["Devi took over billing-api from Meera on 2026-08-20.",
                 "checkout-api calls billing-api.",
                 "The pool ran dry under peak traffic."]:
    change = extract(PROMPT.format(s=sentence), Change)
    print(sentence)
    print("    at:", change.at)
    for edge in change.starts:
        print("    starts:", edge.subject, edge.relation, edge.object)
    for edge in change.ends:
        print("    ends  :", edge.subject, edge.relation, edge.object)
    print()

Devi took over billing-api from Meera on 2026-08-20.
    at: 2026-08-20
    starts: Devi owns billing-api
    ends  : Meera owns billing-api



checkout-api calls billing-api.
    at: None
    starts: checkout-api calls billing-api



The pool ran dry under peak traffic.
    at: None



Three sentences, three different amounts of graph in them.

**The first one is two edges, not one.** "Took over ... from Meera" starts `Devi owns billing-api` *and* ends `Meera owns billing-api`. A schema with a single `edges` list would have recorded the first and silently lost the second, leaving the graph claiming two owners forever. Splitting `starts` from `ends` is the write path admitting what a graph write actually is: not an insert, a **change**.

**The third one produced nothing at all**, and that is the useful result. *"The pool ran dry under peak traffic"* is a real memory — it is journal entry #1, and notebook 2 retrieved it twice. It just has no entities and no relation, so there is nothing to walk. Not every memory belongs in every store, and this is what that looks like from the inside: the extraction runs, costs a model call, and correctly returns empty.

Which is the real price of a graph write, and it is not the latency. `KeyValue.write` cannot be wrong about what it was given. This one **decides** — what counts as an entity, which of four relations fits, which direction it points — and a decision can be made differently on the next run over the same sentence. Notebook 4 comes back to this, because deciding whether a new memory contradicts an existing one is the same problem one level up.

## `update`, and the store that refuses to have one

An update looks like the least interesting operation here. It is the one where the four stores diverge most.

In [6]:
profile.write("editor", "neovim")          # same call as a write: a key holds one value
print("KeyValue  ->", profile.read("editor"))
print()

def score_of(query):
    """Where entry `entry_id` lands for a query, out of the whole journal."""
    return {i: s for s, i, _ in journal.find_like(query, k=len(journal))}[entry_id]

kept = "stale schema cache"                          # both versions say this
dropped = "a migration that skips refreshing the cache"   # only the original said this
was = (score_of(kept), score_of(dropped))

journal.update(entry_id, "PostgREST keeps a schema cache and only refreshes it on NOTIFY.")

now = (score_of(kept), score_of(dropped))
print(f"Journal   -> #{entry_id} vs {kept!r:42} {was[0]:.4f} -> {now[0]:.4f}")
print(f"             #{entry_id} vs {dropped!r:42} {was[1]:.4f} -> {now[1]:.4f}")
print()
print("Graph     ->", "expire(), not update() -- see below")
print("Records   ->", "no update() at all -- see below")

KeyValue  -> neovim



Journal   -> #7 vs 'stale schema cache'                       0.7507 -> 0.7392
             #7 vs 'a migration that skips refreshing the cache' 0.7541 -> 0.6660

Graph     -> expire(), not update() -- see below
Records   -> no update() at all -- see below


**`KeyValue` has no separate update**, and does not need one. A key holds one value; writing it again is the update. That is the whole operation.

**`Journal.update` is a delete and an insert wearing one name.** Look at the two rows: the entry barely moved for the phrase both versions still contain, and dropped hard for the one only the original said. That is what tells you the vector was **replaced** rather than adjusted. There is no operation that nudges an embedding to match edited text, because an embedding is not a description of the text — it is the output of a model that read it. Change the text and the only way to get a matching vector is to call the model again. So every update carries the same network cost, and the same chance of failing, as the original write:

```python
def update(self, entry_id, text):
    self.db.execute("UPDATE entries SET text = ? WHERE id = ?", (text, entry_id))
    self.db.execute("INSERT INTO entries_fts (entries_fts) VALUES ('rebuild')")
    self.db.commit()
    self.vectors.update(ids=[str(entry_id)], documents=[text],
                        embeddings=embed([text], task="RETRIEVAL_DOCUMENT"))
```

Three indexes, all rewritten. Miss one and the entry is findable by half the questions that should find it — which is a bug that never raises, and only shows up as a search that quietly stopped working.

**`Graph` has `expire()` instead**, and that is a deliberate refusal. "Meera owns billing-api" did not become *wrong* on the 20th of August; it stopped being *current*. Editing the edge to say Devi would destroy the answer to "who owned this in July", which is exactly the kind of question a graph is kept for. So the edge gets an end date and stays walkable. The forgetting section comes back to this.

**`Records` has no `update()` and should not.** It is an event log: a row says a deploy failed on the 12th, and that stays true forever. What happens next is a *new row*, not an edit to the old one. A store you can update is a store whose history you can lose, and the one thing a run history is for is being the thing you can trust afterwards.

In [7]:
# housekeeping, so a second run of this notebook starts where the first one did
profile.forget("editor")
journal.forget(entry_id)
records.db.execute("DELETE FROM records WHERE id = ?", (record_id,))
records.db.commit()

print("profile:", profile.data)
print("records:", len(records), "rows")
print("journal:", len(journal), "entries")

profile: {'timezone': 'Asia/Kolkata', 'examples': 'Python'}
records: 6 rows
journal: 6 entries


That `DELETE` reaches past the class on purpose — `Records` offers no way to remove a row, for the reason above. It is here because this notebook has to leave the store as it found it, not because deleting events is something you should do.

## `retrieve`, and a journal big enough to need it

Six entries were enough to settle which index answers which question. They cannot show *ranking*, for a reason worth saying out loud: with six memories you can read all six. Ranking only becomes a problem at a size where the answer has to be chosen without anyone looking at everything.

So the rest of this notebook runs against the same agent's journal a few months later — twenty-four entries across four services, from July to the end of November. Still small enough to print, which is the only reason the numbers below can be checked by eye.

In [8]:
import shutil
import sqlite3
from datetime import date

TODAY = "2026-12-01"

MEMORIES = [
    ("2026-07-02", "checkout-api", "tool_output",
     "The pool ran dry under peak traffic; raising max connections to 60 cleared it."),
    ("2026-07-09", "checkout-api", "concluded",
     "Peak traffic on checkout lands between 19:00 and 21:00 on weekdays."),
    ("2026-08-05", "payments-api", "user_said",
     "Never retry a payment automatically more than twice."),
    ("2026-08-11", "checkout-api", "user_said",
     "Releases go out on Tuesdays unless someone objects."),
    ("2026-08-12", "checkout-api", "tool_output",
     "The migration held a lock on the users table for about forty seconds while it rebuilt an index."),
    ("2026-08-14", "billing-api", "concluded",
     "Billing runs its own migrations on a separate schedule."),
    ("2026-08-18", "billing-api", "tool_output",
     "PostgREST returned PGRST204: column 'invoice_ref' not found in schema cache."),
    ("2026-08-19", "billing-api", "user_said",
     "Priya should be paged when the error rate goes above two percent."),
    ("2026-09-03", "checkout-api", "concluded",
     "Long index rebuilds are the usual cause of checkout latency spikes, not application code."),
    ("2026-09-15", "search-api", "tool_output",
     "Search reindexing pinned all eight CPUs for forty minutes."),
    ("2026-09-22", "billing-api", "tool_output",
     "Billing rolled back 2.9.4 after the error rate hit four percent."),
    ("2026-10-02", "billing-api", "concluded",
     "The schema cache only refreshes on NOTIFY, so a migration that skips it serves a stale column list."),
    ("2026-10-08", "search-api", "concluded",
     "Search can serve stale results for up to an hour after a reindex without anyone noticing."),
    ("2026-10-14", "checkout-api", "tool_output",
     "Checkout p99 latency crossed two seconds for eleven minutes during the 4.4.0 rollout."),
    ("2026-10-21", "payments-api", "tool_output",
     "Payments timed out against the provider sandbox for nine minutes."),
    ("2026-11-02", "payments-api", "concluded",
     "The provider's sandbox is rate limited far below production and is not worth alerting on."),
    ("2026-11-06", "checkout-api", "tool_output",
     "Requests to checkout queued behind a lock on the orders table while a backfill ran."),
    ("2026-11-07", "checkout-api", "concluded",
     "The backfill was the cause; it holds a row lock per batch and the batches were too large."),
    ("2026-11-12", "billing-api", "tool_output",
     "Invoice reconciliation ran for six hours and finished with two hundred unmatched rows."),
    ("2026-11-18", "search-api", "user_said",
     "Devi owns search-api now."),
    ("2026-11-20", "checkout-api", "user_said",
     "Arun wants the backfill batch size capped at five hundred rows from now on."),
    ("2026-11-24", "checkout-api", "tool_output",
     "max connections on checkout is now 200 after the pool was resized."),
    ("2026-11-25", "payments-api", "tool_output",
     "A payments retry storm doubled outbound requests after the provider returned 503s."),
    ("2026-11-28", "billing-api", "concluded",
     "Reconciliation slowness tracks the number of open invoices, not the run window."),
]

MEM = STORE / "memories"


def entry_count(path):
    """Count without opening a Journal, because opening one starts a Chroma client and
    Windows will not let you delete a directory something holds open. The last section of
    this notebook deletes a row for real, so this guard has to be able to rebuild."""
    db = path / "journal.db"
    if not db.exists():
        return 0
    con = sqlite3.connect(db)
    try:
        return con.execute("SELECT COUNT(*) FROM entries").fetchone()[0]
    finally:
        con.close()


if entry_count(MEM) != len(MEMORIES):     # seed once; embeddings cost a network call
    shutil.rmtree(MEM, ignore_errors=True)

memories = Journal(MEM)
if len(memories) != len(MEMORIES):
    memories.write(MEMORIES)

print(len(memories), "entries,", MEMORIES[0][0], "to", MEMORIES[-1][0])

24 entries, 2026-07-02 to 2026-11-28


### Similarity alone

Notebook 2's question, against the bigger journal. Nothing but cosine, which is all `find_like` does:

In [9]:
QUERY = "checkout is timing out and the database seems to be blocking"

def dated(hits):
    """find_like returns (similarity, id, text); the date lives in SQLite."""
    return [(sim, eid, memories.get(eid)["at"], text) for sim, eid, text in hits]

for sim, eid, at, text in dated(memories.find_like(QUERY, k=8)):
    print(f"{sim:.4f}  #{eid:<3} {at}  {text[:52]}")

0.7820  #17  2026-11-06  Requests to checkout queued behind a lock on the ord
0.7545  #9   2026-09-03  Long index rebuilds are the usual cause of checkout 
0.7179  #14  2026-10-14  Checkout p99 latency crossed two seconds for eleven 
0.6955  #18  2026-11-07  The backfill was the cause; it holds a row lock per 
0.6939  #22  2026-11-24  max connections on checkout is now 200 after the poo
0.6874  #2   2026-07-09  Peak traffic on checkout lands between 19:00 and 21:
0.6812  #15  2026-10-21  Payments timed out against the provider sandbox for 
0.6670  #5   2026-08-12  The migration held a lock on the users table for abo


A good ranking. `#17` — requests queueing behind a lock while a backfill ran — is the right answer and it is on top.

Two things it cannot do, though, and the rest of this section is those two in order.

1. **It cannot say "I do not have that."** Ask for three and you get three, always.
2. **It has no idea which of these the agent is actually working with.** `#9`, `#14` and `#18` are all plausible; cosine has no way to prefer the one the agent read yesterday.

The first is simpler, so it goes first.

### It always returns k

Ask this journal about something it has never heard of:

In [10]:
for question in ["what is the capital of France", "how do I renew my passport"]:
    print(question)
    for sim, eid, text in memories.find_like(question, k=3):
        print(f"    {sim:.4f}  #{eid:<3} {text[:52]}")
    print()

what is the capital of France


    0.5157  #4   Releases go out on Tuesdays unless someone objects.
    0.4958  #7   PostgREST returned PGRST204: column 'invoice_ref' no
    0.4894  #3   Never retry a payment automatically more than twice.

how do I renew my passport


    0.5479  #4   Releases go out on Tuesdays unless someone objects.
    0.5111  #3   Never retry a payment automatically more than twice.
    0.5056  #6   Billing runs its own migrations on a separate schedu



Three results each, ranked, with no indication that every one of them is nonsense. A dict answers a missing key with `None`; a vector index has no such answer available to it.

But the scores do know. Line up questions this journal *can* answer against questions it cannot:

In [11]:
for question in [QUERY,
                 "why did billing roll back",
                 "what did Arun ask for",
                 "what is the capital of France",
                 "how do I renew my passport"]:
    sim, eid, text = memories.find_like(question, k=1)[0]
    print(f"  {sim:.4f}   {question[:44]:46} {text[:40]}")

  0.7820   checkout is timing out and the database seem   Requests to checkout queued behind a loc


  0.7664   why did billing roll back                      Billing rolled back 2.9.4 after the erro


  0.7170   what did Arun ask for                          Arun wants the backfill batch size cappe
  0.5157   what is the capital of France                  Releases go out on Tuesdays unless someo
  0.5479   how do I renew my passport                     Releases go out on Tuesdays unless someo


**Answerable: 0.72 to 0.78. Unanswerable: 0.52 to 0.55.** A clean gap, so a floor at around `0.6` gives back the property `read` had for free — the ability to return nothing.

One caution, because it is the sort of number that gets copied: `0.6` separates *these* two groups, on *this* journal, with **`gemini-embedding-001`**. Change the model and the band moves. Change the corpus and it moves again. What transfers is the method — ask things you know the answer to, ask things you know it cannot answer, and look at where the two groups land.

### Which of these is the agent actually working with?

Now the second problem. Before any mechanism, the distinction it rests on — because **three different things all look like "time"**, and confusing them is the expensive mistake here.

| The question | What answers it |
|---|---|
| Is this memory **live in what the agent is doing**? | **reinforcement** — how recently and how often it has been *used*. This section. |
| Is this fact **still true**? | **supersession** — replace the memory, or give it a validity range the way `Graph` edges have one. Not a ranking problem at all. |
| **When was it written**? | the `at` column — metadata you filter on, not a ranking signal |

The middle row is the trap. *"What is the current database config?"* feels like a question that wants recency turned up. It does not. The July config entry is not *less relevant* — it is **wrong**, and moving it from second place to fourth still leaves it in the results for the model to read. Wrong facts need deleting or dating, which is the forgetting section. Turning up a weight is not a fix for a false memory.

So the term built here answers only the first row. **It has no opinion about truth.**

Which means it cannot be computed from the `at` column, because being written is not being used. `Journal` records uses separately:

```sql
CREATE TABLE IF NOT EXISTS accesses (
    entry_id INTEGER NOT NULL,
    at       TEXT NOT NULL
);
```

recorded by whatever *consumed* a memory, never by search. Turning up in a top-k is not use; being read is. A memory that keeps surfacing and keeps being ignored should not get to reinforce itself on its own popularity.

Here is the same query against three different working lives:

In [12]:
from recall import Recall

# record_use=False: everything below MEASURES memory rather than using it, and an
# instrument that mutates what it measures makes each cell depend on the ones before
engine = Recall(memories, today=TODAY, record_use=False)

def show(label):
    print(label)
    for score, eid, at, text in engine(QUERY, k=5):
        print(f"    {score:.4f}  #{eid:<3} uses={len(memories.accesses(eid)):<2} {text[:44]}")
    print()

def forget_all_uses():
    memories.db.execute("DELETE FROM accesses")
    memories.db.commit()

forget_all_uses()
show("A. nothing has ever been used")

memories.used([18] * 4, "2026-11-29")      # the agent spent last week on the backfill
show("B. #18 used four times")

forget_all_uses()
memories.used([22] * 4, "2026-11-29")      # ...or on pool sizing instead
show("C. same corpus, #22 used instead")

forget_all_uses()

A. nothing has ever been used
    0.9264  #17  uses=0  Requests to checkout queued behind a lock on
    0.6882  #9   uses=0  Long index rebuilds are the usual cause of c
    0.5627  #22  uses=0  max connections on checkout is now 200 after
    0.5567  #14  uses=0  Checkout p99 latency crossed two seconds for
    0.5048  #18  uses=0  The backfill was the cause; it holds a row l

B. #18 used four times
    0.9113  #17  uses=0  Requests to checkout queued behind a lock on
    0.6854  #9   uses=0  Long index rebuilds are the usual cause of c
    0.5752  #18  uses=4  The backfill was the cause; it holds a row l
    0.5481  #14  uses=0  Checkout p99 latency crossed two seconds for
    0.5394  #22  uses=0  max connections on checkout is now 200 after

C. same corpus, #22 used instead
    0.9113  #17  uses=0  Requests to checkout queued behind a lock on
    0.6854  #9   uses=0  Long index rebuilds are the usual cause of c
    0.5675  #22  uses=4  max connections on checkout is now 200 after
  

Look at the bottom two rows. `#18` and `#22` are **0.0016 apart** on raw cosine — as close to tied as this corpus gets. In **A** neither has been touched and `#22` edges it. In **B** the agent has been reading `#18`, and it moves to third while `#22` falls to fifth. In **C** the usage is swapped and so is the order.

Same question, same twenty-four memories, different working life, different answer.

And `#17` and `#9` do not move in any of the three. That is the property that makes this safe to ship: **reinforcement breaks ties, it does not override relevance.**

It is worth seeing exactly how weak that is, because "tiebreaker" is easy to say and hard to picture:

### Frequency belongs inside λ, not beside it

A memory used forty times should not fade at the same rate as one used once. The tempting way to write that is a second term added to the score — and it is wrong, because it puts frequency in **competition** with recency: ten uses last quarter start outranking one use yesterday, and a memory the agent has not touched in months beats the one it read this morning. That is not what "frequently used things are remembered longer" means.

Frequency belongs in the decay rate:

$$\lambda(f) = \frac{\lambda_0}{1 + \alpha f}$$

`λ₀` is the fastest decay, for a memory used once. `f` is the access count. `α` is a **consolidation factor** — how much protection each use buys.

A half-life is just the reciprocal of a decay rate, so the same statement in the form the code uses is:

$$H(f) = H_0 \,(1 + \alpha f), \qquad A = 0.5^{\,t/H(f)}$$

Decay still runs from the **most recent use**. Frequency only sets how fast it runs. One signal says *when did this last matter*, the other *how firmly is it held*, and they multiply rather than trade.

Base `0.5` is not a modelling choice — `0.5^(t/H)` and `e^(-λt)` are the same curve with `λ = ln2/H`. It just makes the knob readable: after `H` days, half as active.

In [13]:
for n in (0, 4, 10, 30, 100):
    forget_all_uses()
    if n:
        memories.used([18] * n, "2026-11-29")
    top3 = [eid for _, eid, _, _ in engine(QUERY, k=3)]
    print("#18 used %-4d times -> half-life %5.0f d, activation %.4f, top three: %s"
          % (n, engine.half_life_of(n), engine.activation(18, memories.get(18)["at"]),
             " ".join("#%d" % i for i in top3)))

forget_all_uses()

#18 used 0    times -> half-life    30 d, activation 0.5743, top three: #17 #9 #14
#18 used 4    times -> half-life    90 d, activation 0.9847, top three: #17 #9 #14
#18 used 10   times -> half-life   180 d, activation 0.9923, top three: #17 #9 #14
#18 used 30   times -> half-life   480 d, activation 0.9971, top three: #17 #9 #14
#18 used 100  times -> half-life  1530 d, activation 0.9991, top three: #17 #9 #14


**A hundred uses is not enough.** `#18` trails `#14` by 0.02 of raw cosine, and no amount of reinforcement closes that at `w_reinforcement = 0.2`.

Two things cap it. Activation saturates — once the half-life is long next to the age, `0.5 ** (t/H)` is already near 1 and further uses buy almost nothing (`0.9847` at four uses, `0.9991` at a hundred). And even a perfect activation of 1.0 can only contribute `0.2` to the score.

So the term has a precise amount of power: it can reorder memories within about `0.002` of each other on cosine, and it cannot rescue one that is `0.02` behind however heavily used. If you want a memory to win regardless of how well it matches, reinforcement is the wrong tool and `pinned` is the right one.

In [14]:
import math

print("H(f) = 30 * (1 + 0.5f)\n")
print("%5s %9s | %s" % ("f", "half-life", "activation A at t = ..."))
print("%5s %9s | %s" % ("", "", "  ".join("%6dd" % t for t in (0, 30, 90, 365, 730))))
for f in (0, 1, 5, 10, 60):
    H = engine.half_life_of(f)
    row = "  ".join("%7.4f" % (0.5 ** (t / H)) for t in (0, 30, 90, 365, 730))
    print("%5d %8.0fd | %s" % (f, H, row))

print("\ndays until a memory drops below 0.01 -- effectively gone:")
for f in (0, 1, 5, 10, 60):
    H = engine.half_life_of(f)
    print("   f=%-3d %6.0f days  (%.1f years)" % (f, H * math.log2(100), H * math.log2(100) / 365))

H(f) = 30 * (1 + 0.5f)

    f half-life | activation A at t = ...
                |      0d      30d      90d     365d     730d
    0       30d |  1.0000   0.5000   0.1250   0.0002   0.0000
    1       45d |  1.0000   0.6300   0.2500   0.0036   0.0000
    5      105d |  1.0000   0.8203   0.5520   0.0899   0.0081
   10      180d |  1.0000   0.8909   0.7071   0.2452   0.0601
   60      930d |  1.0000   0.9779   0.9351   0.7618   0.5804

days until a memory drops below 0.01 -- effectively gone:
   f=0      199 days  (0.5 years)
   f=1      299 days  (0.8 years)
   f=5      698 days  (1.9 years)
   f=10    1196 days  (3.3 years)
   f=60    6179 days  (16.9 years)


Read the bottom block first, because it is the whole argument for putting `f` here. A memory used once is gone in about **six months**. One used sixty times survives **seventeen years**. The tail is long, but *selectively* — earned by use, not handed to everything.

That is the difference from the textbook alternative. ACT-R and the human forgetting literature use a **power law**, which gives every memory a long tail whether or not anyone ever looked at it again. For an agent that is the wrong generosity: forgetting the note nobody reopened is exactly how top-k stays clean. `λ(f)` keeps what got used and lets the rest go.

Now look at the top block sideways. `t = H` gives `0.5` on every row, `t = 2H` gives `0.25`, and so on. The ratio `t/H` is the only thing the formula sees — which means **doubling H is identical to halving t**. Frequency does not change the *shape* of forgetting; it changes how fast the clock runs. Every memory rides the same curve, just at its own speed.

One case this cannot reach, by construction: a memory used **once**, long ago, that genuinely matters. `λ(f)` only knows about usage, so it has nothing to work with — and it should not guess. That case is a `pinned` flag, set deliberately, and the forgetting section builds it.

### Combining the two, and the trap in it

Two numbers, one score. The obvious thing is a weighted sum with an even split. Before writing one, print the terms in separate columns:

In [15]:
rows = dated(memories.find_like(QUERY, k=8))
acts = [engine.activation(eid, at) for _, eid, at, _ in rows]

print("%-5s %-12s %9s %11s   %s" % ("id", "at", "cosine", "activation", "text"))
for (sim, eid, at, text), a in zip(rows, acts):
    print("#%-4d %-12s %9.4f %11.4f   %s" % (eid, at, sim, a, text[:42]))

sims = [s for s, _, _, _ in rows]
print()
print("cosine     spans %.4f .. %.4f   (range %.4f)" % (min(sims), max(sims), max(sims) - min(sims)))
print("activation spans %.4f .. %.4f   (range %.4f)" % (min(acts), max(acts), max(acts) - min(acts)))

id    at              cosine  activation   text
#17   2026-11-06      0.7820      0.5612   Requests to checkout queued behind a lock 
#9    2026-09-03      0.7545      0.1279   Long index rebuilds are the usual cause of
#14   2026-10-14      0.7179      0.3299   Checkout p99 latency crossed two seconds f
#18   2026-11-07      0.6955      0.5743   The backfill was the cause; it holds a row
#22   2026-11-24      0.6939      0.8507   max connections on checkout is now 200 aft
#2    2026-07-09      0.6874      0.0351   Peak traffic on checkout lands between 19:
#15   2026-10-21      0.6812      0.3878   Payments timed out against the provider sa
#5    2026-08-12      0.6670      0.0769   The migration held a lock on the users tab

cosine     spans 0.6670 .. 0.7820   (range 0.1150)
activation spans 0.0351 .. 0.8507   (range 0.8156)


**Cosine moves across a range of about 0.12. Activation moves across nearly 0.8.**

A weight is a share of the decision only if the things it multiplies vary by comparable amounts. Written as `0.5 * similarity + 0.5 * activation`, the second term would be doing roughly seven-eighths of the discriminating and the `0.5` in front of the first would be decoration. Cosine over a real corpus lives in a narrow band well away from zero; a decay curve is built to sweep the whole of 0 to 1. They were never on the same scale.

So rescale each term across the candidates before weighting — smallest to 0, largest to 1:

In [16]:
from recall import unit

nsim, nact = unit(sims), unit(acts)
print("%-5s %8s %8s %11s %8s %8s" % ("id", "cos", "cos_n", "activation", "act_n", "score"))
for (sim, eid, at, text), a, ns, na in zip(rows, acts, nsim, nact):
    print("#%-4d %8.4f %8.4f %11.4f %8.4f %8.4f" % (eid, sim, ns, a, na, 0.8 * ns + 0.2 * na))

id         cos    cos_n  activation    act_n    score
#17     0.7820   1.0000      0.5612   0.6451   0.9290
#9      0.7545   0.7607      0.1279   0.1138   0.6313
#14     0.7179   0.4425      0.3299   0.3615   0.4263
#18     0.6955   0.2476      0.5743   0.6612   0.3303
#22     0.6939   0.2338      0.8507   1.0000   0.3871
#2      0.6874   0.1774      0.0351   0.0000   0.1419
#15     0.6812   0.1232      0.3878   0.4325   0.1851
#5      0.6670   0.0000      0.0769   0.0513   0.0103


Now a weight means what it says. But look at the `cos_n` column: something scored **1.0000** — and it will, on every query, including the ones about French capitals. Min-max manufactures a perfect score out of whatever it is handed, because the best candidate *defines* the top of the scale.

Which settles an ordering question. **The floor has to run on the raw score, before normalising**, because normalising destroys the only number that could tell an answerable question from an unanswerable one. Filter first, rescale second — the reverse silently makes every query look answered.

### Over-fetching

`recall.py` puts the steps in order: **over-fetch → floor on raw similarity → normalise → weight**.

Over-fetching means asking the index for more candidates than you intend to keep. It is nearly free — one query, the same graph walk — and it matters because the scoring *reorders*. Ask for exactly `k` and score those `k`, and all the scoring can do is shuffle what similarity already chose; a memory ranked seventh on cosine can never reach the top three however live it is:

In [17]:
for over_fetch in (1, 3):
    e = Recall(memories, today=TODAY, over_fetch=over_fetch, record_use=False)
    pool = [eid for _, eid, _ in memories.find_like(QUERY, k=3 * over_fetch)]
    top3 = [eid for _, eid, _, _ in e(QUERY, k=3)]
    print("over_fetch=%d  candidate pool (%d): %s"
          % (over_fetch, len(pool), " ".join("#%d" % i for i in pool)))
    print("              returned  (3): %s" % " ".join("#%d" % i for i in top3))
    print()

over_fetch=1  candidate pool (3): #17 #9 #14
              returned  (3): #17 #9 #14

over_fetch=3  candidate pool (9): #17 #9 #14 #18 #22 #2 #15 #5 #23
              returned  (3): #17 #9 #14



With `over_fetch=1` the pool *is* the answer — the scoring can reorder those three and nothing else, so a memory sitting fourth on cosine is unreachable no matter what its history is. With `over_fetch=3` nine memories are in play.

The returned three are the same either way, and that is worth stating rather than hiding: **on this corpus, at `w_reinforcement=0.2`, over-fetching never changes the output.** It cannot, for the reason just measured — reinforcement is too weak to promote anything from fourth place.

It stays in because the cost is one query against the same graph, and the case it guards against is real at other settings: raise the reinforcement weight, lengthen the history, or lower `k`, and the pool boundary becomes the thing deciding what you are allowed to see. Paying nothing to remove a hard ceiling is a reasonable trade even when the ceiling is not currently binding.

### Choosing the two weights

Neither weight is a free choice, and they are validated by different tests, because they do different jobs.

**`w_similarity` is validated against questions whose answers you already know.** Six of them, and a sweep:

In [18]:
CHECKS = [                                    # question -> the entry a person would pick
    ("checkout is timing out and the database seems to be blocking", 17),
    ("why did billing roll back", 11),
    ("what did Arun ask for", 21),
    ("what is the connection pool size on checkout", 22),
    ("how long can search results be stale", 13),
    ("when is checkout busiest", 2),
]

print("%-7s %s" % ("w_sim", "  ".join("%-8s" % q[:8] for q, _ in CHECKS)))
for w in (0.5, 0.6, 0.7, 0.8, 0.9):
    e = Recall(memories, today=TODAY, w_similarity=w, w_reinforcement=1 - w,
               record_use=False)
    marks = []
    for question, want in CHECKS:
        ids = [eid for _, eid, _, _ in e(question, k=3)]
        place = ids.index(want) + 1 if want in ids else 0
        marks.append({1: "top", 0: "MISS"}.get(place, "#%d" % place))
    print("%-7.1f %s" % (w, "  ".join("%-8s" % m for m in marks)))

w_sim   checkout  why did   what did  what is   how long  when is 


0.5     top       #2        top       top       top       #3      
0.6     top       top       top       top       top       top     
0.7     top       top       top       top       top       top     
0.8     top       top       top       top       top       top     
0.9     top       top       top       top       top       top     


**An even split gets two of the six wrong.** Everything from `0.6` upward gets all six right. That rules out the number almost everyone starts with, and it is the entire justification for `0.8` — six questions cannot separate `0.8` from `0.9`, so what the sweep establishes is *"similarity must dominate"* and nothing more precise.

Note what this test is blind to. Reinforcement does not change *what the right answer is*; it changes which of several genuinely relevant memories surfaces. A known-answer test cannot see that by construction. So `w_reinforcement` needs its own check, and the thing to check is the failure mode it invites — **rich get richer.** Reinforce whatever you retrieve, and a memory that surfaces gets used, which makes it surface more:

In [19]:
PROBE = ["checkout is timing out and the database seems to be blocking",
         "what causes checkout latency",
         "why did checkout slow down during a rollout"]

for w_reinf in (0.2, 0.5):
    forget_all_uses()
    e = Recall(memories, today=TODAY, w_similarity=1 - w_reinf,
               w_reinforcement=w_reinf, record_use=False)
    before = {q: e(q, k=3)[0][1] for q in PROBE}
    for _ in range(5):                       # five rounds of "use what you retrieved"
        for q in PROBE:
            got = e(q, k=3)
            if got:
                memories.used([got[0][1]], "2026-11-29")
    after = {q: e(q, k=3)[0][1] for q in PROBE}

    used = sorted(((len(memories.accesses(i)), i) for i in range(1, 25)
                   if memories.accesses(i)), reverse=True)
    print("w_reinforcement = %.1f" % w_reinf)
    print("   first place:  %s  ->  %s"
          % (" ".join("#%d" % before[q] for q in PROBE),
             " ".join("#%d" % after[q] for q in PROBE)))
    print("   uses:         %s" % ", ".join("#%d=%d" % (i, n) for n, i in used))
    print()

forget_all_uses()

w_reinforcement = 0.2
   first place:  #17 #9 #14  ->  #17 #9 #9
   uses:         #9=10, #17=5

w_reinforcement = 0.5
   first place:  #17 #23 #23  ->  #17 #17 #17
   uses:         #17=15



At `0.2` the uses stay spread across more than one memory and first place is where similarity put it. At `0.5` a single memory takes first place on **every** query and collects **every** access in the corpus — including questions it was not the best answer to.

So there are two independent reasons for a small reinforcement weight: the sweep rules out an even split on correctness, and the runaway rules it out on stability. Damping `f` to `ln(1+f)` does *not* prevent this — measured, it produces identical rankings here — because the runaway is a property of the weight, not of the shape of λ.

### Every knob, and how honest each one is

| | Job | How it was set |
|---|---|---|
| `floor = 0.6` | the only step that can return nothing | **measured** — answerable 0.72–0.78 vs unanswerable 0.52–0.55 |
| `w_similarity = 0.8` | how much meaning decides | **measured** — sweep over six known answers |
| `w_reinforcement = 0.2` | how much working life decides | **measured twice** — the sweep, and the runaway |
| `over_fetch = 3` | how many candidates the scoring can reach | reasoned — cheap, and it changes reachability |
| `H₀ = 30 days` | how long a once-used memory stays warm | **guessed** |
| `α = 0.5` | what one use buys in extra warm time | **guessed** |

The last two are worth flagging rather than hiding. A journal spanning five months cannot distinguish a 30-day half-life from a 60-day one — every access is recent compared to either, so `t/H` stays near zero and the curves overlap. The same reason the linear-versus-damped question could not be settled here. Those two numbers need a corpus with years in it, and this one has months.

## Two-stage recall, and when not to build it

Everything so far returns whole memories. That is fine here and it is worth seeing why, because the alternative is a real amount of machinery and most stores should not have it.

In [20]:
lengths = [len(r["text"]) for r in memories.db.execute("SELECT text FROM entries")]
print("journal entries: %d, from %d to %d characters (median %d)"
      % (len(lengths), min(lengths), max(lengths), sorted(lengths)[len(lengths) // 2]))
print()
for score, eid, at, text in engine(QUERY, k=3):
    print(f"#{eid:<3} {len(text):>3} chars  {text}")

journal entries: 24, from 25 to 99 characters (median 78)

#17   83 chars  Requests to checkout queued behind a lock on the orders table while a backfill ran.
#9    89 chars  Long index rebuilds are the usual cause of checkout latency spikes, not application code.
#14   85 chars  Checkout p99 latency crossed two seconds for eleven minutes during the 4.4.0 rollout.


A whole retrieve is about two hundred and fifty characters. **A summary of one of these entries would be the entry.** There is nothing to preview, and a function that opened them in full would hand back what you already had.

So: two-stage recall is a solution to a problem this journal does not have. It starts to matter when a memory is not a sentence but a **session** — a debugging run, a postmortem, a design argument — where one memory is a page and three of them will not fit anywhere useful.

In [21]:
SESSIONS = [
    ("2026-11-06", "checkout-api", "tool_output", """\
Paged at 19:40 for checkout p99 above two seconds. Started from the load balancer and worked \
down: no change in request volume, no change in error rate, so not a traffic event. Application \
logs showed handlers sitting in the database driver rather than in our code. pg_stat_activity had \
forty-one sessions in lock waits, all on the orders table. Traced the blocker to a backfill job \
started at 19:12 by the data team, rewriting order_status in batches. The batches were sized at \
fifty thousand rows, which holds a row lock long enough that ordinary checkout writes pile up \
behind it. Asked them to stop the job at 19:58 and latency recovered within ninety seconds. \
Nothing was wrong with checkout itself. Left a note for the data team that batch size is the \
lever here, not the schedule, since running it overnight would only move the problem to whoever \
is awake then."""),
    ("2026-11-12", "billing-api", "tool_output", """\
Invoice reconciliation ran for six hours and eleven minutes, against a usual forty minutes. \
Checked the obvious things first: no schema change, no index dropped, query plans unchanged. \
The run processes open invoices one at a time and the count of open invoices has grown from \
about nine hundred to just over eleven thousand since August, because the collections workflow \
stopped closing invoices that go to manual review. So the job is not slower, there is simply \
far more of it. Two hundred rows finished unmatched, all of them manual-review invoices with no \
payment reference. The fix is upstream in collections, not in reconciliation, and the run time \
will keep climbing until the open-invoice count stops growing. Flagged for Meera."""),
    ("2026-08-18", "billing-api", "tool_output", """\
PostgREST began returning PGRST204 on invoice inserts, saying column 'invoice_ref' was not found \
in the schema cache. The column exists in the database and the migration that added it completed \
successfully. PostgREST caches the schema at startup and refreshes it only on a NOTIFY \
pgrst,'reload schema' signal, which our migration tooling does not send. So every migration that \
adds a column leaves PostgREST serving a stale column list until something restarts it. Sent the \
NOTIFY manually and inserts recovered immediately. This will recur on every future migration \
unless the tooling sends it, and the failure mode is confusing because the database is correct \
and only the API layer disagrees."""),
    ("2026-10-14", "checkout-api", "tool_output", """\
Rollout of 4.4.0 pushed checkout p99 over two seconds for eleven minutes. The new release adds \
an availability check per basket line, and the check was written to call the inventory service \
once per line rather than once per basket. A basket of twelve lines therefore made twelve \
sequential calls. Under normal load this is survivable; during the rollout half the fleet was \
still on 4.3.x, so the inventory service saw a mixed traffic pattern it had never been sized \
for. Rolled forward with a batched version rather than back, because rolling back would have \
left baskets created under 4.4.0 in an inconsistent state. Latency returned to baseline within \
two minutes of the batched build reaching every node."""),
    ("2026-09-15", "search-api", "tool_output", """\
Reindexing pinned all eight CPUs for forty minutes and search latency roughly tripled while it \
ran. The reindex is single-threaded per shard but we run eight shards, so it saturates whatever \
box it lands on. Nothing is wrong with the job; it is scheduled at 02:00 and normally nobody \
notices. This run started at 11:00 because the previous night's attempt failed on a disk-space \
check and the retry policy is "as soon as possible" rather than "at the next window". The retry \
policy is the bug. Serving stale results for an hour is acceptable and documented; tripling \
latency in the middle of the day is not."""),
    ("2026-11-24", "checkout-api", "tool_output", """\
Resized the checkout connection pool from 60 to 200 after the July incident kept coming up in \
review. The original 60 was chosen when checkout ran on four application nodes; it now runs on \
fourteen, so the per-node share had quietly fallen to a level where any slow query exhausted it. \
Confirmed against pg_stat_activity that the database has headroom for 200 across all clients. \
This does not fix slow queries, it stops them from turning into a full outage while somebody \
looks at them."""),
]

print("%d sessions, %d to %d characters each"
      % (len(SESSIONS), min(len(s[3]) for s in SESSIONS), max(len(s[3]) for s in SESSIONS)))

6 sessions, 487 to 868 characters each


Three of those in a prompt is roughly two thousand four hundred characters of history, for one question. It is affordable once and not affordable every turn, and most of it will be irrelevant — the point of a top-3 is that you are not sure which one you want.

So return something small enough to choose from, and fetch the full text only for what gets chosen. The question is where the small thing comes from.

### The summary is written, not derived at read time

Three options, and only one survives contact.

**Truncate.** Free, no model, and the first eighty characters of a debugging session are the paging alert every time. The informative part of a write-up is usually its conclusion.

**Summarise at read time.** A model call per candidate per query, for text that has not changed since it was written. You would pay it again on the next question.

**Summarise at write time.** Once per memory, ever. So the summary is a column, and it is written by whatever writes the text.

That last decision brings an ordering constraint with it, and getting it backwards is a quiet failure rather than a loud one. `Journal` takes a `summarize` hook so the ordering lives in the store rather than in every caller:

```python
def write(self, entries):
    # every summary first: a model call that fails should leave the journal untouched
    summaries = [self.summarize(text) if self.summarize else None for ... in entries]
    for (at, about, source, text), summary in zip(entries, summaries):
        self.db.execute("INSERT INTO entries (..., text, summary) VALUES (...)", ...)
```

The fallible step runs before the durable one, and both columns land together. Summarise *after* writing and a timeout leaves new text sitting behind an old summary — a memory that describes itself incorrectly, which is worse than one with no summary at all.

For the same reason there is no setter for `summary` on its own. A field you can write independently is a field that can drift from what it describes, and the drift is silent: two-stage recall keeps working, just on a description of something the entry no longer says.

In [22]:
from llm import chat

SUMMARY_PROMPT = """Summarise this memory in one line, under 90 characters.

It will be shown in a list so an agent can decide whether to open the full text, so lead with
what the memory would let someone answer. No preamble, just the line.

Memory:
{t}"""

def one_line(text):
    return chat([{"role": "user", "content": SUMMARY_PROMPT.format(t=text)}]).strip()

SESS = STORE / "sessions"
sessions = Journal(SESS, summarize=one_line)
if len(sessions) != len(SESSIONS):
    del sessions
    shutil.rmtree(SESS, ignore_errors=True)
    sessions = Journal(SESS, summarize=one_line)
    sessions.write(SESSIONS)

for row in sessions.db.execute("SELECT id, summary, text FROM entries ORDER BY id"):
    print("#%-3d %4d -> %3d chars   %s"
          % (row["id"], len(row["text"]), len(row["summary"]), row["summary"]))

#1    868 ->  76 chars   Backfill job locking orders table caused checkout slowdown; stop job to fix.
#2    737 ->  84 chars   Reconciliation slowed because open invoices rose from 900 to 11k, collections issue.
#3    696 ->  83 chars   PostgREST returns PGRST204 on inserts; schema cache not refreshed after migrations.
#4    708 ->  77 chars   4.4.0 rollout caused checkout >2s latency; fixed by batching inventory calls.
#5    607 ->  83 chars   Latency spiked midday because a reindex ran off‑schedule due to a retry‑policy bug.
#6    487 ->  66 chars   How to stop checkout outages? Increase DB pool to 200 connections.


Roughly nine to one, and each line says what the memory would let you answer rather than how it opens. Note the third keeps `NOTIFY pgrst,'reload schema'` — the literal somebody would actually search for. A summary that drops the identifiers is a summary you cannot find anything with.

**The bill this creates lands on the write path.** Every save and every update now costs a model call that a single-stage store does not, forever, whether or not that memory is ever searched. Which is one more reason not to build this until memories are genuinely large: on the twenty-four one-line entries above it would be pure cost with nothing bought.

### Two tools, not a pipeline

The obvious shape is a function that returns summaries and then a second call that parses which ids the model wants. The parsing is the problem — the model replies *"let me look at 3 and 7"* and now you own a small, permanently fragile extractor.

Tool calling already solved that. Two tools, and the model's choice arrives as arguments:

In [23]:
from typing import Annotated

from tools import tool


@tool
def recall(query: Annotated[str, "what to look for, in your own words"]) -> str:
    """Search memory. Returns ids, dates and one-line summaries, not full text."""
    hits = Recall(sessions, today=TODAY, record_use=False)(query, k=3)   # open_memory does it
    if not hits:
        return "Nothing in memory matches that."
    # the date comes from the row, not the summary: the listing has to carry whatever the
    # choice depends on, and "which of these is the November one" is a common way to choose
    return "\n".join(f"#{eid}  {at}  {sessions.get(eid)['summary']}"
                     for _, eid, at, _ in hits)


@tool
def open_memory(memory_id: Annotated[int, "an id from a recall result"]) -> str:
    """Read one memory in full. Call this before answering from a recall result."""
    row = sessions.get(memory_id)
    if row is None:
        return f"No memory #{memory_id}."
    sessions.used([memory_id], TODAY)      # being opened is use; being listed is not
    return row["text"]


print(recall(query="why was checkout slow in November"))

#1  2026-11-06  Backfill job locking orders table caused checkout slowdown; stop job to fix.
#4  2026-10-14  4.4.0 rollout caused checkout >2s latency; fixed by batching inventory calls.
#6  2026-11-24  How to stop checkout outages? Increase DB pool to 200 connections.


Now the loop from chapter 2 runs it, with no memory-specific machinery anywhere — `recall` and `open_memory` are ordinary tools, and `repeat_from` is what lets the model come back for a second turn after reading the summaries:

In [7]:
from act import Act
from chat import Conversation
from observe import Observe
from runtime import Runtime
from think import Think

from memory_tools import EXTRACTORS

TOOLS = [recall, open_memory]

# a question the summaries cannot answer: the batch size and the recovery time are in the
# full text of #1 and nowhere else, so the second stage is not optional here
QUESTION = ("When checkout slowed on the 6th of November, what batch size was the backfill "
            "using, and how long did latency take to recover once it was stopped?")

sessions.db.execute("DELETE FROM accesses")          # so "what got opened" is this run only
sessions.db.commit()

conversation = Conversation()
Runtime(
    loop=[
        Observe(event=QUESTION),
        Think(tools=[t.schema for t in TOOLS], allow_fork=False),
        Act(functions={t.name: t.function for t in TOOLS},
            schemas={t.name: t.args for t in TOOLS}),
        # without this the model never sees what the tools returned -- and without
        # EXTRACTORS it sees a model's one-sentence rewrite of them, which drops the ids
        Observe(extractors=EXTRACTORS),
    ],
    repeat_from=1,
    max_iterations=16,
).run(conversation)

called = [m["tool"] for m in conversation.messages
          if m.get("role") == "decision" and m.get("tool")]
opened = [i for i in range(1, len(SESSIONS) + 1) if sessions.accesses(i)]

print("tools called:", " -> ".join(called))
print("opened      :", opened, "   (recorded as used; listing recorded nothing)")
print()
print(conversation.messages[-1]["content"])

tools called: recall -> open_memory
opened      : [1]    (recorded as used; listing recorded nothing)

The backfill was using batches of 50,000 rows, and latency recovered within about 90 seconds after the job was stopped.


The model listed, then opened the one it needed, and answered with two facts — the batch size and the recovery time — that appear nowhere in any summary. It was never told about ids, formats or how many to ask for: `open_memory(memory_id=1)` is a tool call, and a tool call is already structured.

**The question was chosen to require the second stage**, which is not a trick so much as the condition for the second stage existing at all. Ask *"why was checkout slow?"* and the summary — *"Backfill job locking orders table caused checkout slowdown; stop job to fix"* — already answers it, so the model answers from the listing and never opens anything. Measured over three runs, it did exactly that. **That is correct behaviour**, and it is worth sitting with: if your summaries are good, most questions never reach the second stage, and the token saving is the whole return on the write-time cost.

One caveat on what you see above, because notebook outputs make things look more certain than they are: it is one run. Ten runs of this cell *without* `EXTRACTORS` went straight through seven times. The other three took a detour — guessed `open_memory(memory_id=0)`, passed a date as the id, or searched again with the question reworded — and all three traced to one place: the listing had reached the model without its `#1`. It looked like a model that could not tell when to stop. It was a model with nothing to open. With `EXTRACTORS`, five runs of five went `recall -> open_memory -> answer`. That is why the cell prints the tool sequence rather than resting on the final message.

Two things in that cell are corrections rather than decoration, and two more in the tools above it. All four were failures before they were fixes.

**The trailing `Observe()`.** `Act` does not put a tool's result into the conversation; it leaves it on `conversation.pending`, and `Observe` is what turns that into something the model can read. Leave it out — as the first version of this cell did — and the model calls `recall`, sees nothing come back, and calls `recall` again with slightly different words until the runtime stops it for repeating itself. The tools were fine. The loop was missing the step that closes it.

**`extractors=EXTRACTORS` on that `Observe`.** By default `Observe` has a model restate each tool result as one plain sentence. For `recall` the rewrite kept the entry but usually dropped the `#1` in front of it — the id survived in about one run in five — and the id is the only thing that makes an entry openable. `EXTRACTORS`, from `memory_tools.py`, hands `recall` and `open_memory` results over word for word, with no model call. A listing is already written for the model to read; there is nothing to restate.

**The date in the listing.** The first version showed `#id  summary` and nothing else, and the model opened the entry about the 14th of October when asked about the 6th of November. The summaries were good — they simply did not say *when*, and there is no way to pick the November one out of a list that never mentions dates.

**And one sentence in a docstring.** With `open_memory` described as *"Read one memory in full."*, the model called `recall` five times in a row and never opened anything. Adding *"Call this before answering from a recall result"* was the whole fix. The docstring is not documentation here — `03_tools` sends it to the API as the tool's `description`, so it is the only instruction the model ever gets about **when** a tool applies. `recall` returning ids is not a hint the model is obliged to take; something has to say that ids are for opening.

That is the same lesson as keeping `PGRST204` in a summary, arriving from the other direction: **the listing is what the choice is made from, so it has to carry whatever the choice depends on.** A summary is a compression of the text; the date was never in the text to begin with, it was in the row. Stage one is a rendering job, not just a summarising one.

**And `open_memory` does a second job that is easy to miss.** It records the access:

```python
sessions.used([memory_id], TODAY)      # being opened is use; being listed is not
```

Which settles a question the reinforcement section left open: *who writes to the `accesses` table?* The rule is that **the last stage records**, and the two arrangements put it in different places.

**Two-stage:** `open_memory` records, and `Recall` is built with `record_use=False`. A memory listed and skipped decays exactly as if it had never come up, because nothing wrote anything down when it was listed. That is the sharp signal — *something read this* — and it is only available because there is a second stage to be the last one.

**Single-stage:** the retrieve is the last stage, so it records what it returned. `Recall` does this by default; there is nothing downstream that could. The signal is coarser — it says *the ranking kept choosing this*, not *something read it* — and for short memories that is a fair trade, because the returned line **is** the whole memory and there was never anything further to read. Being returned is as close to being used as that store can get.

What you must not do is neither. A single-stage store built with `record_use=False` has an `accesses` table that stays empty forever, so every activation falls back to the write date and the reinforcement term quietly degenerates into the write-date decay this section spent its first half arguing against. It keeps working, it keeps producing plausible numbers, and it has stopped measuring what it claims to.

### When not to build it

Worth stating plainly, because everything above is machinery:

| | Two-stage |
|---|---|
| memories are a sentence or two | **no** — the summary is the memory, and `open_memory` returns what you already had |
| memories are sessions, transcripts, documents | **yes** — a top-k does not fit, and choosing is cheap |
| you need usage to be meaningful | **yes** — it is the only place an honest "this was used" signal exists |
| writes are frequent and searches are rare | **think again** — the cost is per write, and it is paid whether or not anything is ever recalled |

The journal from the rest of this notebook stays single-stage, and should.

## Writing without contradicting yourself

`update` arrived early in this notebook as a *mechanism* — change the text, re-embed, rewrite the summary. Nothing has yet decided to call it.

That gap has a specific shape. An agent finishes a run, concludes something, and writes it down. It does not know what it already remembers, so everything it concludes is a new entry. Do that for a month and the journal holds the same fact five times, three of them out of date, and a retrieve returns whichever the ranking happens to prefer.

The failure is not that a wrong memory exists. It is that **both versions exist**, and once they do the store cannot tell you which is current — that information was thrown away at write time, by writing instead of updating.

So a write is not one operation. It is a search, then a decision, then one of three actions:

| | when | effect |
|---|---|---|
| **add** | nothing remembered covers this | a new entry |
| **update** | something says this, with a value this replaces | the old text is **gone** — one entry, current |
| **ignore** | something already says this | nothing |

`update` is what keeps contradictions from accumulating, because it replaces in place rather than writing alongside.

### The decision needs the retrieve half of this notebook

The search that decides a write is the same search that answers a question. To know whether a conclusion is new, you have to find what resembles it — which is `Recall`, pointed at the thing about to be written rather than at a user's question.

Then a model classifies the relationship. `03_tools` supplies the shape: `Literal` puts the three actions in the schema so the model picks rather than invents, and a `model_validator` catches the one incoherent combination — an `update` that does not say what it is updating.

In [25]:
from typing import Literal, Optional

from pydantic import BaseModel, Field, model_validator


class WriteDecision(BaseModel):
    """What to do with something the agent just concluded."""

    action: Literal["add", "update", "ignore"]
    target_id: Optional[int] = Field(
        default=None, description="which existing memory, when the action is update")
    reason: str = Field(description="one short sentence")

    @model_validator(mode="after")
    def update_needs_a_target(self):
        if self.action == "update" and self.target_id is None:
            raise ValueError("action 'update' requires target_id: which memory is replaced?")
        if self.action != "update" and self.target_id is not None:
            raise ValueError(f"target_id is only meaningful for 'update', not '{self.action}'")
        return self


DECIDE = """An agent has just concluded something. Decide what to do with it, given what it
already remembers.

New conclusion:
    {candidate}

Existing memories that resemble it:
{related}

  add     nothing remembered covers this
  update  one of them states the same thing with a value or detail this replaces -- name which
  ignore  one of them already says this

Return JSON matching the schema."""


def decide(candidate, k=3):
    """Search first, then classify. Returns (decision, the memories it was shown)."""
    # a duplicate check reads memories to compare against; that is not using them
    related = Recall(memories, today=TODAY, record_use=False)(candidate, k=k)
    if not related:
        return WriteDecision(action="add", reason="nothing in the journal resembles it"), []
    block = "\n".join(f"    #{eid}  {text}" for _, eid, _, text in related)
    return extract(DECIDE.format(candidate=candidate, related=block), WriteDecision), related

The validator earns its place here rather than decorating the schema. `action="update"` with no `target_id` is a decision the calling code cannot execute — it says *replace something* without saying what — and it is exactly the shape a model produces when it half-commits. Raising sends the real message back and the model fills it in; the alternative is a `None` reaching `journal.update(None, text)`.

Three conclusions, one of each kind:

In [26]:
CANDIDATES = [
    "Backfill batches should be capped at five hundred rows.",
    "Arun has lowered the backfill batch size cap to two hundred rows.",
    "The search reindex retry policy fires immediately instead of waiting for the next window.",
]

for candidate in CANDIDATES:
    decision, related = decide(candidate)
    print(candidate)
    print("   saw:      ", " ".join(f"#{eid}" for _, eid, _, _ in related))
    print(f"   decision:  {decision.action:<7} target={decision.target_id}  ({decision.reason})")
    print()

Backfill batches should be capped at five hundred rows.
   saw:       #21 #18 #17
   decision:  ignore  target=None  (Already captured in memory #21)



Arun has lowered the backfill batch size cap to two hundred rows.
   saw:       #21 #18 #17
   decision:  update  target=21  (new cap replaces previous 500 rows)



The search reindex retry policy fires immediately instead of waiting for the next window.
   saw:       #13 #10 #23
   decision:  add     target=None  (No existing memory captures this immediate retry behavior.)



Measured over three runs each, those three are stable: `ignore`, `update` naming `#21`, and `add`.

A fourth is not, and it is the honest one to show. Offer it *"On 2 July, raising max connections to 60 cleared the pool exhaustion"* — which restates `#1` and adds a date — and it splits between `ignore` and `update #1` across runs.

That is not a bug being papered over. "Is a date worth rewriting an entry for?" has no correct answer, so the model does not have one either. What matters is what it **never** chose: `add`. On the ambiguous case the procedure degrades toward not duplicating, which is the direction you want it to fail in.

### What this actually costs, and what it does not fix

**Every write now costs a retrieve and a model call.** Together with the summary from the previous section, an agent writing a memory makes three round trips before anything is stored. That is the price of a journal that stays consistent, and it is worth knowing before agreeing to it — for a store nothing ever searches, it buys nothing.

**The check is only as good as the search.** This is the part worth dwelling on. `decide()` classifies against whatever `Recall` returned, so a memory the search fails to surface may as well not exist: the model sees nothing resembling the candidate, answers `add`, and the duplicate is created by the same procedure that exists to prevent it.

Which puts the floor from the retrieve section under a second, opposite pressure. For a user's question, returning nothing is a feature — better silence than the least-bad of five bad candidates. For a duplicate check, returning nothing is read as **permission**, and a floor set too high quietly manufactures the duplicates it was meant to catch.

Two consequences worth taking seriously:

- The `k` for a write check and the `k` for answering a question are different numbers. A question wants the best three; a duplicate check wants everything that could plausibly be the same fact, and would rather look at ten.
- The floor for a write check should be lower than the one for recall, or absent. Being shown an irrelevant memory costs the classifier nothing — it answers `add` anyway. Being shown nothing costs correctness.

**And it does not fix a memory that was simply wrong.** Nothing here checks a conclusion against the world; it checks it against what is already stored. An agent confidently wrong twice in the same direction produces a consistent journal that is consistently wrong, and this procedure will happily `ignore` the second one for agreeing with the first.

## `forget`

Two things wear this name, and only one of them removes anything.

**Decay** changes where a memory ranks. The row stays in the table, keeps costing index and storage, and stays eligible for every query.

**Deletion** removes it, and is optional.

Start with what decay looks like across the whole journal. Nothing here has been used, so what is on screen is pure age:

In [27]:
forget_all_uses()

print("%-5s %-12s %11s   %s" % ("id", "written", "activation", "text"))
for row in memories.db.execute("SELECT * FROM entries ORDER BY id"):
    print("#%-4d %-12s %11.6f   %s"
          % (row["id"], row["at"], engine.activation(row["id"], row["at"]), row["text"][:50]))

id    written       activation   text
#1    2026-07-02      0.029839   The pool ran dry under peak traffic; raising max c
#2    2026-07-09      0.035077   Peak traffic on checkout lands between 19:00 and 2
#3    2026-08-05      0.065456   Never retry a payment automatically more than twic
#4    2026-08-11      0.075189   Releases go out on Tuesdays unless someone objects
#5    2026-08-12      0.076947   The migration held a lock on the users table for a
#6    2026-08-14      0.080586   Billing runs its own migrations on a separate sche
#7    2026-08-18      0.088388   PostgREST returned PGRST204: column 'invoice_ref' 
#8    2026-08-19      0.090454   Priya should be paged when the error rate goes abo
#9    2026-09-03      0.127922   Long index rebuilds are the usual cause of checkou
#10   2026-09-15      0.168794   Search reindexing pinned all eight CPUs for forty 
#11   2026-09-22      0.198425   Billing rolled back 2.9.4 after the error rate hit
#12   2026-10-02      0.250000   The s

A ladder from about `0.03` for the oldest entries to `0.93` for the newest. `#1` and `#2`, from the first week of July, have decayed to roughly three percent of a fresh memory.

So ask a question that only their neighbourhood can answer:

In [28]:
for score, eid, at, text in engine("when is checkout busiest", k=3):
    print(f"{score:.4f}  #{eid:<3} {at}  activation={engine.activation(eid, at):.4f}  {text[:44]}")

0.8000  #2   2026-07-09  activation=0.0351  Peak traffic on checkout lands between 19:00
0.4463  #9   2026-09-03  activation=0.1279  Long index rebuilds are the usual cause of c
0.3859  #17  2026-11-06  activation=0.5612  Requests to checkout queued behind a lock on


**A memory at a few percent activation comes back first.**

That is decay working as designed, and it is the thing to be clear about. Decay lowers a *score*. It does not remove a row, and it does not lower that row's standing against memories which are not in the store. When a decayed entry is the closest thing the journal has to a question, a low score is still the highest score.

So decay by itself never makes the store smaller. Everything ever written stays, ranked further and further down. Whether that is a problem is a decision, not a fact.

### Keep everything, or set a bound

`Recall` takes an `evict_below` figure, and it is **`None` by default — nothing is ever deleted for having decayed.** Memories sink in the ranking as they go unused, and stay.

That is a reasonable permanent answer, not just a safe starting one. A journal of a few thousand entries is a couple of megabytes and a search that is still instant; there is nothing worth reclaiming, and anything you keep is something you can still be asked for.

The case for a bound shows up when the store grows faster than the questions do — an agent writing notes every run, for years — where the cost is not disk but **dilution**: every retrieve is a top-k, and a k full of memories nothing has needed since spring is a k that has no room for the ones it has.

Setting `evict_below` turns activation into that bound. `stale()` reports what falls under it:

In [29]:
for threshold in (0.001, 0.01, 0.05):
    victims = engine.stale(threshold)
    print("stale(%.3f): %2d of %d  ->  %s"
          % (threshold, len(victims), len(memories),
             " ".join("#%d" % i for _, i in victims) or "nothing"))

stale(0.001):  0 of 24  ->  nothing
stale(0.010):  0 of 24  ->  nothing
stale(0.050):  2 of 24  ->  #1 #2


**At `0.01` it matches nothing at all**, because the oldest memory here has only reached `0.0298`. Five months is not long enough for anything to die at a thirty-day half-life. Like the floor and the weights, the number comes off your own corpus — a store with years in it would evict at `0.01` constantly, and one a fortnight old should evict at nothing.

At `0.05` two entries qualify, and the pair is worth looking at before deleting either.

- **`#1`** — *"raising max connections to 60 cleared it"*. July, never returned to.
- **`#2`** — *"peak traffic on checkout lands between 19:00 and 21:00"*. Also July, also never returned to.

Activation cannot separate them, because it only measures **use**, and neither has been used. But `#2` is the kind of thing nobody asks until the week they need it badly. *Unused* and *not worth keeping* are not the same property, and no threshold can tell them apart.

Which is what `pin()` is for, and why a sweep reports before it deletes:

In [30]:
memories.pin(2)                              # not asked about; still worth having
print("after pinning #2 ->", [i for _, i in engine.stale(0.05)])

before = len(memories)
print("dry run          ->", [i for _, i in engine.sweep(0.05, dry_run=True)],
      f"| journal still {len(memories)} entries")

removed = engine.sweep(0.05, dry_run=False)
print("for real         ->", [i for _, i in removed],
      f"| journal now {len(memories)} entries (was {before})")
print("gone from Chroma too:", memories.vectors.get(ids=["1"])["ids"] == [])

memories.pin(2, False)

after pinning #2 -> [1]
dry run          -> [1] | journal still 24 entries
for real         -> [1] | journal now 23 entries (was 24)
gone from Chroma too: True


`sweep()` reports by default and deletes only when told, because there is no undo and the list is the part worth reading twice.

It also has to reach **both** indexes. `Journal.forget` removes the SQLite row and the Chroma vector together — miss one and the entry stays findable by half the questions that should find it, which raises no error and surfaces only as a search that quietly stopped working.

Re-running this notebook restores the journal: the seeding cell checks the count and rebuilds when it does not match. A section that deletes things needs the section above it to be reproducible.

### On a graph, a lifetime instead

`Graph` has no `forget()`, and adding one would be a mistake.

*"Meera owns billing-api"* is not something to remove when Devi takes over on the twentieth of August. It is something that was the case for a period, and the period has ended. Delete the edge and *who owned this in July* becomes unanswerable; overwrite it with Devi and the graph will answer that question confidently and wrongly.

So the edge gets an end date and stays walkable:

In [31]:
org.expire("Meera", "owns", "billing-api", at="2026-08-20")

if not any(e["subject"] == "Devi" for e in org.about("billing-api")):
    org.write("Devi", "owns", "billing-api", since="2026-08-20")

for when in ("2026-07-01", "2026-09-01", None):
    owners = [e["subject"] for e in org.about("billing-api", at=when)
              if e["relation"] == "owns"]
    print(f"owners of billing-api at {str(when):12} -> {owners}")

owners of billing-api at 2026-07-01   -> ['Meera']
owners of billing-api at 2026-09-01   -> ['Devi']
owners of billing-api at None         -> ['Meera', 'Devi']


Three questions, three correct answers, out of a graph nothing was destructively edited in. July returns Meera, September returns Devi, and `at=None` — *do not filter by time* — returns both, because both describe real periods.

That is the contrast to take out of this section. A journal can only keep a memory or delete it, so bounding it means **losing** what it drops. A graph can say *when* something held, so the same operation costs nothing: the edge stops being current without stopping being available.

Which of those you get is decided by the store, and the store was decided in notebook 2 by what would be asked of it. "Who owns this now" and "who owned this in July" are both questions about the same edge, and only one shape of store can answer both.

## What you built

✓ Separated **read** from **retrieve** — a read is exact and can say no, a retrieve is ranked and cannot — then found that only one of the four stores ranks at all. SQL orders by a column and a graph returns the chain it walked; neither produces a score, and neither is missing one

✓ Timed the same verb across three stores — `1.1 ms`, `4.0 ms`, `733.8 ms` — and hit a fourth that cannot be called from a sentence at all, because `Graph.write` takes a triple and something has to produce it

✓ Found that one sentence is **two** edges: *"Devi took over billing-api from Meera"* starts one and ends another, and a schema with a single list would have left the graph claiming two owners forever

✓ Watched a real memory produce **no** edges — *"the pool ran dry under peak traffic"* is journal entry #1, retrieved twice in notebook 2, and has nothing to walk. The extraction ran, cost a call, and correctly returned nothing

✓ Proved a vector update is a replacement rather than an edit, by watching one entry barely move for a phrase both versions kept (`0.7507 → 0.7392`) and drop hard for one only the original had (`0.7541 → 0.6660`)

✓ Gave `Records` no `update()` on purpose, because an event log's whole value is that you cannot edit it, and `Graph` an `expire()` instead of one

✓ Measured where a floor belongs rather than picking one: answerable questions land at **0.72–0.78**, unanswerable ones at **0.52–0.55**, and that gap lives in the **raw** cosine which min-max destroys — so the only step that can say *I do not have that* has to run before the rescaling

✓ Separated three things that all look like time: **reinforcement** (has this been used), **validity** (is it still true), and the **write date** (metadata to filter on). Only the first is a ranking signal, and confusing it with the second is how a stale fact gets nudged down a list it is still on

✓ Recorded uses in an `accesses` table and watched the same query return a different third place depending on what the agent had been reading — while `#17` and `#9` never moved, because reinforcement breaks ties and does not override relevance

✓ Measured exactly how weak that is: `#18` trails `#14` by 0.02 of cosine and **a hundred uses does not close it**, because activation saturates at `0.9991` and `0.2 × 1.0` is the most the term can ever contribute

✓ Put frequency **inside** lambda rather than beside it — `λ(f) = λ₀/(1+αf)` — so it stretches the clock instead of competing with recency. A memory used sixty times survives seventeen years; a one-off dies in six months

✓ Printed the two terms in separate columns before trusting a weighted sum, and found cosine moving `0.115` against decay's `0.816`, which made "fifty-fifty" seven-eighths recency in practice

✓ Measured the weights instead of choosing them. Six questions with known answers rule out the even split — it gets two of them wrong — and at `w_reinforcement = 0.5` a single memory takes first place on **every** query and collects **every** access in the corpus

✓ Built two-stage recall only after showing it should not exist for one-line memories, where the summary *is* the entry, then compressed six session write-ups about nine to one and let the model open what it wanted

✓ Fixed four things that looked right and were not: a loop missing its trailing `Observe()`, so tool results never reached the model; a listing with no dates, so a question about November opened October; a docstring that never said what ids were for; and a demo question the summaries already answered

✓ Derived `record_use` from the store rather than leaving it a flag — **the last stage records** — because a single-stage store with nothing recording has an empty `accesses` table and a reinforcement term that has quietly become write-date decay

✓ Gave the write path a decision — **add / update / ignore** — so the agent searches before it writes and `update` finally has something that calls it, with a `model_validator` rejecting an `update` that names nothing to update

✓ Said plainly what that check cannot do: it is bounded by the search, so a memory retrieval fails to surface becomes a duplicate created by the very procedure meant to prevent it — and it compares a conclusion against the store, never against the world

✓ Separated decay from deletion, and watched a memory at `0.0351` activation still come back **first**, because decay lowers a score and does not remove a row

✓ Made eviction a policy that is **off by default**, then found `0.01` matches nothing on a five-month journal while `0.05` catches one entry worth losing and one that is merely never asked about — which is what `pin()` and a dry run are for

✓ Ended on the graph, where forgetting is neither decay nor deletion: an edge gets an end date, so July returns Meera, September returns Devi, and `at=None` returns both

✓ Left three numbers explicitly unearned — `H₀ = 30 days`, `α = 0.5`, and the linear-versus-damped choice — because a journal spanning five months cannot separate them. Every access is recent next to either half-life, so the curves overlap. They need a corpus with years in it, and this one has months

**Next:** `04_types_of_memory.ipynb` — semantic, episodic and procedural memory: which store each belongs in, and what breaks when they are paired wrong. The write path is where they diverge hardest, and the decision built here is the example: **add / update / ignore** is right for a fact and destroys a run history, because *"deploy failed on the 3rd"* and *"deploy failed on the 12th"* look like duplicates to a similarity check and are two events.